# PAR (Pedestrian Attribute Recognition)

This notebook simulates a neural network for Pedestrian Attribute Recognition (PAR) using PyTorch.

### Dataset info

The MIVIA PAR Dataset consists of 105,244 images (93,082 in the training set and 12,162 in the validation set) annotated with the following labels (or part of them):

```
0) Image: name of the image file

1) Color of the upper clothes: the considered values are black, blue, brown, gray, green, orange, pink, purple, red, white, and yellow and are represented, in this order, with the labels [1,2,3,4,5,6,7,8,9,10,11].

2) Color of the lower clothes: same as above.

3) Gender: the considered values are male and female, represented in this order with the values [0,1].

4) Bag: we consider the absence or presence of a bag, representing it with the values [0,1].

5) Hat: we consider the absence or presence of a hat, representing it with the values [0,1].

The value -1 marks the unavailability of the specific annotation.
```



To address the issue of missing annotations in the dataset, Visual Question Answering (VQA) was previously employed. In this context, VQA was used to generate missing annotations based on questions and answers related to the dataset images. As a result, the annotations used in the dataset do not contain any -1 values.

# Step 0: Imports, device


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import csv, os, random
from PIL import Image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

C:\Users\orlan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Step 1: Data Loading and Preprocessing

In [2]:

# Custom dataset class for training data
class CustomTrainDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file1, root_dir1, transform=None):
        self.data = []
        self.transform = transform

        # Load data from CSV file and root directory
        for CSV_FILE, ROOT_DIR in [(csv_file1, root_dir1)]:
            if CSV_FILE is not None:
                with open(CSV_FILE, 'r') as file:
                    reader = csv.reader(file)
                    i = 0
                    for row in reader:   # Extract labels and images
                        # Extract gender label (4th column, so index 3)
                        gender_label = int(row[3])
                        
                        # Extract bag label (5th column, so index 4)
                        bag_label = int(row[4])
                        
                        # Extract hat label (6th column, so index 5)
                        hat_label = int(row[5])

                        # Extract image filename and construct image path
                        img_name = row[0]
                        img_path = os.path.join(ROOT_DIR, img_name)
                        
                        # Append tuple of image path and labels to data list
                        self.data.append((img_path, gender_label, bag_label, hat_label))

    def __len__(self):
        # Return the total number of samples
        return len(self.data)

    def __getitem__(self, idx):
        # Get image path and labels for the given index
        img_path, gender_label, bag_label, hat_label = self.data[idx]
        
        # Open image and convert it to RGB format
        image = Image.open(img_path).convert('RGB')
        
        # Apply transformations if any
        if self.transform:
            image = self.transform(image)
        
        # Return image and labels
        return image, gender_label, bag_label, hat_label


# Custom dataset class for validation data
class CustomValDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file1, root_dir1, transform=None):
        self.data = []
        self.transform = transform

        # Load data from CSV file and root directory
        for CSV_FILE, ROOT_DIR in [(csv_file1, root_dir1)]:
            if CSV_FILE is not None:
                with open(CSV_FILE, 'r') as file:
                    reader = csv.reader(file)
                    i = 0
                    for row in reader:   # Extract labels and images
                        # Extract gender label (4th column, so index 3)
                        gender_label = int(row[3])
                        
                        # Extract bag label (5th column, so index 4)
                        bag_label = int(row[4])
                        
                        # Extract hat label (6th column, so index 5)
                        hat_label = int(row[5])
                        
                        # Extract image filename and construct image path
                        img_name = row[0]
                        img_path = os.path.join(ROOT_DIR, img_name)
                        
                        # Append tuple of image path and labels to data list
                        self.data.append((img_path, gender_label, bag_label, hat_label))

    def __len__(self):
        # Return the total number of samples
        return len(self.data)

    def __getitem__(self, idx):
        # Get image path and labels for the given index
        img_path, gender_label, bag_label, hat_label = self.data[idx]
        
        # Open image and convert it to RGB format
        image = Image.open(img_path).convert('RGB')
        
        # Apply transformations if any
        if self.transform:
            image = self.transform(image)
        
        # Return image and labels
        return image, gender_label, bag_label, hat_label


# Define transformations to be applied to the images
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Resize images to 224x224
    transforms.ToTensor(),   # Convert images to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])   # Normalize images
])

## Step 2: Model Definition

In [3]:
# ResNet-based model, with custom output heads

import torch
import torchvision.models as models
import torch.nn as nn

#resnet = models.resnet18(pretrained=True)   # ResNet18
resnet = models.resnet34(pretrained=True)   # ResNet34
backbone = nn.Sequential(*list(resnet.children())[:-1])   # remove output layer

# task-specific heads

class GenderClassifier(nn.Module):
    def __init__(self):
        super(GenderClassifier, self).__init__()
        self.fc = nn.Linear(resnet.fc.in_features, 1)   # output layer for binary gender classification

    def forward(self, x):
        x = self.fc(x)
        return x

class BagClassifier(nn.Module):
    def __init__(self):
        super(BagClassifier, self).__init__()
        self.fc = nn.Linear(resnet.fc.in_features, 1)   # output layer for binary bag classification

    def forward(self, x):
        x = self.fc(x)
        return x

class HatClassifier(nn.Module):
    def __init__(self):
        super(HatClassifier, self).__init__()
        self.fc = nn.Linear(resnet.fc.in_features, 1)   # output layer for binary hat classification

    def forward(self, x):
        x = self.fc(x)
        return x

# create model by combining ResNet backbone with task-specific heads

class CombinedModel(nn.Module):
    def __init__(self):
        super(CombinedModel, self).__init__()
        self.backbone = backbone
        self.gender_classifier = GenderClassifier()
        self.bag_classifier = BagClassifier()
        self.hat_classifier = HatClassifier()

    def forward(self, x):
        features = self.backbone(x).squeeze()   # extract features from ResNet backbone
        gender_output = self.gender_classifier(features)   # predict gender
        bag_output = self.bag_classifier(features)   # predict bag
        hat_output = self.hat_classifier(features)   # predict hat
        return gender_output, bag_output, hat_output


C:\Users\orlan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\orlan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Step 3: Model, Loss Function and Optimizer

In [4]:
combined_model = CombinedModel().to(device)

# define separate loss functions for each task
criterion_gender = nn.BCELoss()   # Binary Cross-Entropy Loss for gender classification
criterion_bag = nn.BCELoss()   # Binary Cross-Entropy Loss for bag classification
criterion_hat = nn.BCELoss()   # Binary Cross-Entropy Loss for hat classification

# define optimizer for the whole combined model (including ResNet backbone and task-specific heads)
optimizer = optim.Adam(combined_model.parameters(), lr=0.001)

In [5]:
# Create training dataset and data loader
train_dataset = CustomTrainDataset(csv_file1='dataset/new_mivia/annotation/training_set.txt', root_dir1='dataset/new_mivia/data/training_set', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
print("Trainset size:", len(train_dataset), "\n")

# Create validation dataset and data loader
val_dataset = CustomValDataset(csv_file1='dataset/new_mivia/annotation/validation_set.txt', root_dir1='dataset/new_mivia/data/validation_set', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print("Validation set size:", len(val_dataset), "\n")


Trainset size: 93081 

Validation set size: 12162 



In [6]:
# training for fine-tuning the entire combined model

def train_combined_model(model, train_loader, criterion_gender, criterion_bag, criterion_hat, optimizer, num_epochs=1, curr_epoch=1):
    for epoch in range(num_epochs):
        model.train()   # set model to training mode
        running_loss_gender = 0.0
        running_loss_bag = 0.0
        running_loss_hat = 0.0

        i = 1
        print("Total batches:", len(train_loader), "\n\nProcessing:")

        for inputs, gender_labels, bag_labels, hat_labels in train_loader:
            if i % 50 == 0:
                print(str(i))
            else:
                print(str(i), end=" ")
            i += 1
            inputs, gender_labels, bag_labels, hat_labels = inputs.to(device), gender_labels.to(device), bag_labels.to(device), hat_labels.to(device)

            optimizer.zero_grad()   # put gradients to zero

            # forward pass
            gender_output, bag_output, hat_output = model(inputs)

            # reshape and apply sigmoid activation for binary classifications
            gender_output = gender_output.view(-1, 1)
            gender_output = torch.sigmoid(gender_output)

            bag_output = bag_output.view(-1, 1)
            bag_output = torch.sigmoid(bag_output)

            hat_output = hat_output.view(-1, 1)
            hat_output = torch.sigmoid(hat_output)

            # compute losses
            gender_loss = criterion_gender(gender_output.squeeze(1), gender_labels.float())
            bag_loss = criterion_bag(bag_output.squeeze(1), bag_labels.float())
            hat_loss = criterion_hat(hat_output.squeeze(1), hat_labels.float())

            # compute total loss
            total_loss = gender_loss + bag_loss + hat_loss

            # backward pass and optimization
            total_loss.backward()
            optimizer.step()

            # update running losses
            running_loss_gender += gender_loss.item() * inputs.size(0)
            running_loss_bag += bag_loss.item() * inputs.size(0)
            running_loss_hat += hat_loss.item() * inputs.size(0)

        # compute average epoch losses
        epoch_loss_gender = running_loss_gender / len(train_loader.dataset)
        epoch_loss_bag = running_loss_bag / len(train_loader.dataset)
        epoch_loss_hat = running_loss_hat / len(train_loader.dataset)

        # print epoch statistics
        print(f"\n\nEpoch [{epoch + 1}/{num_epochs}]\nGender Loss: {epoch_loss_gender:.4f}\n"
              f"Bag Loss: {epoch_loss_bag:.4f}\nHat Loss: {epoch_loss_hat:.4f}\n\n")

        # save model
        torch.save(model.state_dict(), "model_RN34_SD_"+str(curr_epoch)+".pth")


## Step 5: Validation

In [9]:
# VALIDATION WITH SAMPLES DISPLAYING

from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np



gender_mapping = {0: 'male', 1: 'female'}
bh_mapping = {0: 'false', 1: 'true'}


# define evaluation function for the combined model
def evaluate_combined_model(model, val_loader, criterion_gender, criterion_bag, criterion_hat):
    model.eval()   # set the model to evaluation mode

    running_loss_gender = 0.0
    running_loss_bag = 0.0
    running_loss_hat = 0.0
   
    correct_predictions_gender = 0
    correct_predictions_bag = 0
    correct_predictions_hat = 0
    total_predictions = 0
    # for F1-score
    
    all_true_gender = []
    all_pred_gender = []
    all_true_bag = []
    all_pred_bag = []
    all_true_hat = []
    all_pred_hat = []

    with torch.no_grad():   # disable gradient calculation during evaluation
        i = 1
        print("Total batches:", len(val_loader), "\n\nProcessing:")
        for inputs, gender_labels, bag_labels, hat_labels in val_loader:
            if i % 50 == 0:
                print(str(i))
            else:
                print(str(i), end=" ")
            i += 1
            inputs, gender_labels, bag_labels, hat_labels = inputs.to(device), gender_labels.to(device), bag_labels.to(device), hat_labels.to(device)

            gender_labels = gender_labels.float()   # convert gender labels to float
            bag_labels = bag_labels.float()   # convert bag labels to float
            hat_labels = hat_labels.float()   # convert hat labels to float

            # forward pass
            gender_output, bag_output, hat_output = model(inputs)

            # apply sigmoid activation for binary classifications
            gender_output = torch.sigmoid(gender_output.squeeze())
            bag_output = torch.sigmoid(bag_output.squeeze())
            hat_output = torch.sigmoid(hat_output.squeeze())

            # compute losses
            
            gender_loss = criterion_gender(gender_output.squeeze(), gender_labels.squeeze())
            bag_loss = criterion_bag(bag_output.squeeze(), bag_labels.squeeze())   # adjust labels for BCELoss
            hat_loss = criterion_hat(hat_output.squeeze(), hat_labels.squeeze())   # adjust labels for BCELoss

            # update running losses
            
            running_loss_gender += gender_loss.item() * inputs.size(0)
            running_loss_bag += bag_loss.item() * inputs.size(0)
            running_loss_hat += hat_loss.item() * inputs.size(0)

            # compute correct predictions for gender classification
            predictions_gender = (gender_output > 0.5).squeeze().long()   # convert probabilities to binary predictions
            correct_predictions_gender += (predictions_gender == gender_labels).sum().item()
            all_true_gender.extend(gender_labels.cpu().numpy())
            all_pred_gender.extend(predictions_gender.cpu().numpy())

            # compute correct predictions for bag classification
            predictions_bag = (bag_output > 0.5).squeeze().long()   # convert probabilities to binary predictions
            correct_predictions_bag += (predictions_bag == bag_labels).sum().item()
            all_true_bag.extend(bag_labels.cpu().numpy())
            all_pred_bag.extend(predictions_bag.cpu().numpy())

            # compute correct predictions for hat classification
            predictions_hat = (hat_output > 0.5).squeeze().long()   # convert probabilities to binary predictions
            correct_predictions_hat += (predictions_hat == hat_labels).sum().item()
            all_true_hat.extend(hat_labels.cpu().numpy())
            all_pred_hat.extend(predictions_hat.cpu().numpy())

            total_predictions += inputs.size(0)


            # display a random sample from the batch
            index = np.random.randint(inputs.size(0))   # select a random sample from the batch
            input_sample = inputs[index].cpu()   # move the input sample to CPU and detach from gradients
           
            true_gender = gender_labels[index].item()
            predicted_gender = predictions_gender[index].item()
            true_bag = bag_labels[index].item()
            predicted_bag = predictions_bag[index].item()
            true_hat = hat_labels[index].item()
            predicted_hat = predictions_hat[index].item()

            # convert input sample to numpy array and transpose to (H, W, C) format
            input_sample = input_sample.numpy().transpose((1, 2, 0))

            # display the image with true and predicted labels
            # plt.imshow(input_sample)
            # plt.title(
            #           f"Gender: {gender_mapping[true_gender]}, Predicted: {gender_mapping[predicted_gender]}\n"
            #           f"Bag: {bh_mapping[true_bag]}, Predicted: {bh_mapping[predicted_bag]}\n"
            #           f"Hat: {bh_mapping[true_hat]}, Predicted: {bh_mapping[predicted_hat]}\n"
            #           )
            # plt.axis('off')
            # plt.show()


    # compute average epoch losses
    
    epoch_loss_gender = running_loss_gender / len(val_loader.dataset)
    epoch_loss_bag = running_loss_bag / len(val_loader.dataset)
    epoch_loss_hat = running_loss_hat / len(val_loader.dataset)

    # compute accuracy for each task
    
    accuracy_gender = correct_predictions_gender / total_predictions
    accuracy_bag = correct_predictions_bag / total_predictions
    accuracy_hat = correct_predictions_hat / total_predictions

    # compute F1-score for each task
    
    f1_score_gender = f1_score(all_true_gender, all_pred_gender)
    f1_score_bag = f1_score(all_true_bag, all_pred_bag)
    f1_score_hat = f1_score(all_true_hat, all_pred_hat)

    # print evaluation metrics
    
    print(f"GENDER  ---  Loss: {epoch_loss_gender:.4f}, Accuracy: {accuracy_gender:.4f}, F1-score: {f1_score_gender:.4f}")
    print(f"BAG  ---  Loss: {epoch_loss_bag:.4f}, Accuracy: {accuracy_bag:.4f}, F1-score: {f1_score_bag:.4f}")
    print(f"HAT  ---  Loss: {epoch_loss_hat:.4f}, Accuracy: {accuracy_hat:.4f}, F1-score: {f1_score_hat:.4f}")

In [10]:
### TRAIN AND VALIDATE
DONE = 0
UP_TO = 10

for i in range(DONE, UP_TO):
    print("\n\n------------------------------   EPOCH " + str(i+1) + "   ------------------------------\n\n")
    train_combined_model(combined_model, train_loader, criterion_gender, criterion_bag, criterion_hat, optimizer, num_epochs=1, curr_epoch=i+1)
    evaluate_combined_model(combined_model, val_loader, criterion_gender, criterion_bag, criterion_hat)



------------------------------   EPOCH 1   ------------------------------


Total batches: 2909 

Processing:
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50
51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100
101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150
151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200
201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 

KeyboardInterrupt: 

## Step 6: Save the model (optional)

In [ ]:
# NOTE: pay attention to eventual overwrites!!!
torch.save(combined_model.state_dict(), '/content/drive/MyDrive/model_RN34_SD.pth')
torch.save(combined_model, '/content/drive/MyDrive/model_RN34.pth')